# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined using a Croissant schema accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the latest mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset from schema
dataset = mlc.Dataset(croissant_url)

# Access and print metadata (as object attributes, not subscripting)
print(f"Name: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}\n\nVersion: {dataset.metadata.version}")
if hasattr(dataset.metadata, 'keywords'):
    print(f"\nKeywords: {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview

Review available record sets, their fields, field `@id`s, and data columns.

The list below shows each record set's `@id`, human-readable name, and its available fields and columns. Always reference entities by their `@id` for reproducibility.

In [ ]:
# List all record sets and their fields by @id

record_sets = list(dataset.record_sets())  # Returns list of RecordSet objects
if not record_sets:
    print("No record sets found in metadata. This may indicate the schema needs update or must be accessed differently.")
else:
    for recset in record_sets:
        print(f"Record Set @id: {recset['@id']}")
        print(f"  Name: {recset.get('name','(no name)')}")
        print("  Fields:")
        fields = recset.get('field', [])
        # Normalize to list
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            fid = f['@id'] if '@id' in f else (f if isinstance(f, str) else 'N/A')
            print(f"    Field @id: {fid}")
        print("  Columns:")
        columns = recset.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        for c in columns:
            cid = c['@id'] if '@id' in c else (c if isinstance(c, str) else 'N/A')
            print(f"    Column @id: {cid}")
        print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

_**Note:** For this dataset, the record set(s) may be referenced only by their `@id`. We'll retrieve all accessible record sets and load them into Pandas DataFrames by their `@id`._

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
if not record_set_ids:
    print("No record sets available in this dataset.")
else:
    dataframes = {}
    for recset_id in record_set_ids:
        # Load all records as list-of-dict
        try:
            records = list(dataset.records(record_set=recset_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[recset_id] = df
                print(f"Loaded {df.shape[0]} records for record set @id: {recset_id}")
            else:
                print(f"No records found for record set @id: {recset_id}")
        except Exception as e:
            print(f"Failed to load records for {recset_id}: {e}")
            continue

    # Display columns and preview the first non-empty DataFrame
    for recset_id, df in dataframes.items():
        print(f"\nFirst 5 rows for record set @id: {recset_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
        break  # Show only the first one

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering rows, normalizing numeric fields, and grouping data by attributes. All references to record sets and fields use their `@id`s as per Croissant best practices.

_The particular field names/ids below should be replaced as needed based on the fields available in each record set._

In [ ]:
# Select a record set and numeric field for EDA.
if not dataframes:
    print('No dataframes loaded for EDA.')
else:
    # Use the first loaded record set as example
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"EDA on record set @id: {record_set_id}")
    # Try to find a likely numeric field (column) with integer/float values
    numeric_candidates = df.select_dtypes(include=['int','float']).columns.tolist()
    if not numeric_candidates:
        print('No numeric fields detected.')
    else:
        numeric_field_id = numeric_candidates[0]  # Pick the first numeric field as an example
        print(f"Using numeric field: {numeric_field_id}")
        # Apply filter: values > threshold
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        mean_val = filtered_df[numeric_field_id].mean()
        std_val = filtered_df[numeric_field_id].std()
        col_norm = f'{numeric_field_id}_normalized'
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - mean_val) / std_val if std_val > 0 else 0
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, col_norm]].head())

        # Grouping: Try to find a categorical field to group by
        category_candidates = df.select_dtypes(include=['object']).columns.tolist()
        # Exclude all-nan fields and non-informative fields
        category_candidates = [c for c in category_candidates if not df[c].isnull().all() and df[c].nunique() < len(df)/2]
        if category_candidates:
            group_field_id = category_candidates[0]
            print(f"\nGrouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print('No suitable categorical field found for grouping.')

## 5. Visualization

Visualize data distributions or relationships using the selected fields and groupings. All axes and legends should reference field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization example: histogram and group comparison
if not dataframes:
    print('No dataframes loaded for visualization.')
else:
    df = dataframes[record_set_id]
    if not numeric_candidates:
        print('No numeric fields available for plotting.')
    else:
        fig, axs = plt.subplots(1, 2, figsize=(14, 5))
        # Histogram of numeric field
        sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, ax=axs[0])
        axs[0].set_title(f"Distribution of {numeric_field_id}")
        axs[0].set_xlabel(numeric_field_id)
        axs[0].set_ylabel("Count")

        # If grouping field exists, boxplot
        if category_candidates:
            group_field_id = category_candidates[0]
            sns.boxplot(data=df, x=group_field_id, y=numeric_field_id, ax=axs[1])
            axs[1].set_title(f"{numeric_field_id} by {group_field_id}")
            axs[1].set_xlabel(group_field_id)
            axs[1].set_ylabel(numeric_field_id)
        else:
            axs[1].axis('off')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:

- Load dataset metadata from a Croissant schema using the `mlcroissant` library.
- Inspect available record sets, fields, and their `@id`s for robust referencing.
- Extract tabular data by record set `@id` and convert to Pandas DataFrame.
- Conduct simple EDA: filtering, normalization, grouping by key fields using their `@id`.
- Visualize the data distributions and groupwise summaries.

Replace or extend the exploration steps with domain-specific analyses using the field and record set `@id`s above for reproducibility.